In [ ]:
import re
import requests
import pandas as pd
from datetime import date, timedelta

In [ ]:
INPUT_EXCEL  = "fund_list.xlsx"
OUTPUT_EXCEL = "fund_nav_output.xlsx"

DAYS_BACK = 365     # 近一年
SORT_DESC = True    # 輸出日期新到舊
TIMEOUT = 30

In [ ]:
def fmt_ymd(d: date) -> str:
    """API 常見格式：YYYY-M-D（不補零）"""
    return f"{d.year}-{d.month}-{d.day}"

def fetch_bcd_raw(endpoint: str, a: str, start: str, end: str, b: int = 1, timeout: int = 30) -> str:
    base = f"https://fund.bot.com.tw/w/bcd/{endpoint}.djbcd"
    params = {"a": a, "b": b, "c": start, "d": end}
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(base, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.text.strip()

def parse_bcd_nav(raw: str) -> pd.DataFrame:
    raw = raw.strip()
    parts = re.split(r"\s+", raw, maxsplit=1)
    if len(parts) < 2:
        raise ValueError("BCD 回傳無法分成兩段（日期段/數值段）")

    dates_csv, nav_csv = parts[0], parts[1]
    dates = [x for x in dates_csv.split(",") if x]
    navs  = [x for x in nav_csv.split(",") if x]

    if len(dates) == 0 or len(navs) == 0:
        raise ValueError("日期或淨值序列為空")
    if len(dates) != len(navs):
        raise ValueError(f"日期/淨值筆數不一致：dates={len(dates)} navs={len(navs)}")

    df = pd.DataFrame({
        "日期": pd.to_datetime(dates, format="%Y%m%d", errors="coerce"),
        "淨值": pd.to_numeric(navs, errors="coerce")
    }).dropna()

    if df.empty:
        raise ValueError("解析後 DataFrame 為空")

    # 先由舊到新，方便算 diff
    df = df.sort_values("日期").reset_index(drop=True)
    return df

def detect_fund_type_and_fetch(a: str, start: str, end: str, timeout: int = 30) -> tuple[str, pd.DataFrame]:
    # 先試國內 tBCDNavList（國內基金淨值頁會用到 tBCDNavList）[2](https://fund.bot.com.tw/w/wb/wb02a.djhtm?a=FLZK1)
    try:
        raw = fetch_bcd_raw("tBCDNavList", a=a, start=start, end=end, timeout=timeout)
        df = parse_bcd_nav(raw)
        return "國內", df
    except Exception:
        pass

    # 再試海外 BCDNavList
    raw = fetch_bcd_raw("BCDNavList", a=a, start=start, end=end, timeout=timeout)
    df = parse_bcd_nav(raw)
    return "海外", df

def compute_change_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["漲/跌"] = df["淨值"].diff()
    df["漲跌幅(%)"] = (df["漲/跌"] / df["淨值"].shift(1)) * 100
    return df

In [ ]:
# 用 dtype=str 盡量避免 pandas 自動把代碼轉成數字
fund_list = pd.read_excel(INPUT_EXCEL, engine="openpyxl", dtype=str).fillna("")

required_cols = ["基金代號", "基金名稱", "查詢傳輸代號"]
missing = [c for c in required_cols if c not in fund_list.columns]
if missing:
    raise ValueError(f"Excel 缺少欄位：{missing}。請確認欄名需為：{required_cols}")

# 去空白，避免 " ACDS134 " 這種情況
for c in required_cols:
    fund_list[c] = fund_list[c].astype(str).str.strip()

# 移除完全空白列
fund_list = fund_list[(fund_list["基金代號"] != "") & (fund_list["查詢傳輸代號"] != "")]
fund_list.head()

In [ ]:
end_date = date.today()
start_date = end_date - timedelta(days=DAYS_BACK)
start = fmt_ymd(start_date)
end   = fmt_ymd(end_date)
print("查詢區間：", start, "～", end)

all_rows = []
fail_rows = []

for _, r in fund_list.iterrows():
    fund_code = r["基金代號"]
    fund_name = r["基金名稱"]
    trans_a   = r["查詢傳輸代號"]

    try:
        fund_type, nav_df = detect_fund_type_and_fetch(a=trans_a, start=start, end=end, timeout=TIMEOUT)
        nav_df = compute_change_cols(nav_df)

        # 填回你要的欄位
        nav_df.insert(0, "查詢傳輸代號", trans_a)
        nav_df.insert(0, "基金名稱", fund_name)
        nav_df.insert(0, "基金代號", fund_code)
        nav_df.insert(3, "基金類型", fund_type)

        # 輸出順序（新到舊）
        if SORT_DESC:
            nav_df = nav_df.sort_values("日期", ascending=False).reset_index(drop=True)

        all_rows.append(nav_df)
        print(f"[OK] {fund_code} | {trans_a} | {fund_type} | rows={len(nav_df)}")

    except Exception as e:
        fail_rows.append({"基金代號": fund_code, "基金名稱": fund_name, "查詢傳輸代號": trans_a, "錯誤": str(e)})
        print(f"[FAIL] {fund_code} | {trans_a} -> {e}")

result_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
fail_df   = pd.DataFrame(fail_rows)
result_df.head(10)

In [ ]:
cols = ["基金代號", "基金名稱", "查詢傳輸代號", "基金類型", "日期", "淨值", "漲/跌", "漲跌幅(%)"]
out_df = result_df[cols].copy() if not result_df.empty else result_df

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    out_df.to_excel(writer, index=False, sheet_name="nav")
    if not fail_df.empty:
        fail_df.to_excel(writer, index=False, sheet_name="fail")

OUTPUT_EXCEL